# Домашнее задание № 4. Языковые модели

## Задание 1 (8 баллов).

В семинаре для генерации мы использовали предположение маркова и считали, что слово зависит только от 1 предыдущего слова. Но ничто нам не мешает попробовать увеличить размер окна и учитывать два или даже три прошлых слова. Для них мы еще сможем собрать достаточно статистик и, логично предположить, что качество сгенерированного текста должно вырасти.

Попробуйте сделать языковую модель, которая будет учитывать два предыдущих слова при генерации текста.
Сгенерируйте несколько текстов (3-5) и расчитайте перплексию получившейся модели. 
Можно использовать данные из семинара или любые другие (можно брать только часть текста, если считается слишком долго). Перплексию рассчитывайте на 10-50 отложенных предложениях (они не должны использоваться при сборе статистик).


Подсказки:  
    - нужно будет добавить еще один тэг \<start>  
    - можете использовать тот же подход с матрицей вероятностей, но по строкам хронить биграмы, а по колонкам униграммы 
    - тексты должны быть очень похожи на нормальные (если у вас получается рандомная каша, вы что-то делаете не так)
    - у вас будут словари с индексами биграммов и униграммов, не перепутайте их при переводе индекса в слово - словарь биграммов будет больше словаря униграммов и все индексы из униграммного словаря будут формально подходить для словаря биграммов (не будет ошибки при id2bigram[unigram_id]), но маппинг при этом будет совершенно неправильным 

In [1]:
import re, random, math
from collections import Counter

START = "<s>"
END = "</s>"

def tokenize(s):
    s = re.sub(r"([.,!?;:()\"'])", r" \1 ", s)
    return [t.lower() for t in s.split() if t.strip()]

text = """
Alice was beginning to get very tired of sitting by her sister on the bank,
and of having nothing to do. Once or twice she had peeped into the book her
sister was reading. There was nothing very remarkable in that. Suddenly a
White Rabbit with pink eyes ran close by her. Alice started to her feet.
"""
text = text * 10

sentences = re.split(r'(?<=[.!?])\s+', text)
sentences = [s.strip() for s in sentences if s.strip()]

held_out = 10
train_sentences = sentences[:-held_out]
test_sentences = sentences[-held_out:]

unigram = Counter()
trigram = Counter()
vocab = set()

for s in train_sentences:
    toks = [START, START] + tokenize(s) + [END]
    for i in range(len(toks)):
        unigram[toks[i]] += 1
        vocab.add(toks[i])
        if i >= 2:
            trigram[(toks[i-2], toks[i-1], toks[i])] += 1

V = len(vocab)

def prob(w1, w2, w3, k=1):
    num = trigram.get((w1, w2, w3), 0) + k
    denom = sum(trigram.get((w1, w2, w), 0) for w in vocab) + k * V
    return num / denom

def generate(max_len=40):
    w1, w2 = START, START
    out = []
    for _ in range(max_len):
        ws = list(vocab)
        ps = [prob(w1, w2, w) for w in ws]
        r = random.random()
        cum = 0
        for w, p in zip(ws, ps):
            cum += p
            if r <= cum:
                nxt = w
                break
        if nxt == END:
            break
        out.append(nxt)
        w1, w2 = w2, nxt
    txt = " ".join(out)
    txt = re.sub(r"\s+([.,!?])", r"\1", txt)
    return txt.capitalize()

def sentence_logprob(tokens):
    toks = [START, START] + tokens + [END]
    s = 0
    n = 0
    for i in range(2, len(toks)):
        p = prob(toks[i-2], toks[i-1], toks[i])
        s += math.log(p)
        n += 1
    return s, n

total_log = 0
total_n = 0

for s in test_sentences:
    lp, n = sentence_logprob(tokenize(s))
    total_log += lp
    total_n += n

perplexity = math.exp(-total_log / total_n)

print("Perplexity:", perplexity)
print("\nGenerated texts:\n")
for _ in range(5):
    print(generate(), "\n")


Perplexity: 6.278331685198038

Generated texts:

Alice having remarkable beginning alice to and a the very suddenly to peeped sitting on on suddenly her she having in into in <s> get once suddenly or in ran that to sister remarkable to there, with that remarkable 

On bank suddenly bank she having 

Twice. a reading rabbit beginning twice there suddenly <s> bank a alice nothing sister, or into <s> nothing rabbit of ran book alice white on started to of into book reading book nothing suddenly nothing started into started 

Nothing feet, pink. ran on nothing <s> sister beginning had book was she ran started suddenly alice on to beginning a remarkable on pink. remarkable and in or remarkable do had into by 

Alice the that white there suddenly, that in with with on 



## Задание № 2* (2 балла). 

Измените функцию generate_with_beam_search так, чтобы она работала с моделью, которая учитывает два предыдущих слова. 
Сравните получаемый результат с первым заданием. 
Также попробуйте начинать генерацию не с нуля (подавая \<start> \<start>), а с какого-то промпта. Но помните, что учитываться будут только два последних слова, так что не делайте длинные промпты.